In [15]:
import pandas as pd
import numpy as np
import os

# 1. 设置文件路径 (相对路径)
BASE_PATH = 'raw data'

def reduce_mem_usage(df):
    """
    遍历所有列，修改数据类型以减少内存使用。
    (已修复：自动跳过 datetime 时间类型，防止报错)
    """
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'📉 原始内存占用: {start_mem:.2f} MB')

    for col in df.columns:
        col_type = df[col].dtype
        
        # 核心修复：如果列是 object (文字) 或者包含 'datetime' (时间)，直接跳过不处理
        if col_type != object and 'datetime' not in str(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            
            # 处理整数
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            # 处理浮点数
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2
    print(f'✅ 优化后内存占用: {end_mem:.2f} MB')
    print(f'🚀 内存减少了: {100 * (start_mem - end_mem) / start_mem:.1f}%')
    return df

In [16]:
# 2. 加载数据
print("⏳ 正在加载数据...")
train = pd.read_csv(os.path.join(BASE_PATH, 'train.csv'))
test = pd.read_csv(os.path.join(BASE_PATH, 'test.csv'))
stores = pd.read_csv(os.path.join(BASE_PATH, 'stores.csv'))
oil = pd.read_csv(os.path.join(BASE_PATH, 'oil.csv'))
transactions = pd.read_csv(os.path.join(BASE_PATH, 'transactions.csv'))
holidays = pd.read_csv(os.path.join(BASE_PATH, 'holidays_events.csv'))

# 3. 解决油价缺失问题 (The 43 missing values)
# 先把日期转成时间格式
oil['date'] = pd.to_datetime(oil['date'])
# 生成完整日期索引 (填补周末的日期空缺)
date_range = pd.date_range(start=oil['date'].min(), end=oil['date'].max())
oil = oil.set_index('date').reindex(date_range).reset_index()
oil.rename(columns={'index': 'date', 'dcoilwtico': 'oil_price'}, inplace=True)
# 线性插值填补空缺 (Interpolate)
oil['oil_price'] = oil['oil_price'].interpolate(method='linear', limit_direction='both')
print("✅ 油价数据清洗完毕 (缺失值已填补)")

# 4. 合并数据 (Merge)
train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])

# 给train和test打上标签，方便以后分开
train['type'] = 'train'
test['type'] = 'test'
test['sales'] = np.nan # 占位符

# 拼在一起
print("⏳ 正在合并 Train 和 Test...")
df = pd.concat([train, test], sort=False).reset_index(drop=True)

# 把 Store 信息和 Oil 信息贴上去
print("⏳ 正在关联 Store 和 Oil 信息...")
df = df.merge(stores, on='store_nbr', how='left')
df = df.merge(oil, on='date', how='left')

# 5. 最后的内存优化
print("⏳ 正在进行最终内存优化...")
df = reduce_mem_usage(df)

# 6. 检查结果
print(f"\n🎉 阶段1完成！最终数据形状: {df.shape}")
display(df.head())

⏳ 正在加载数据...
✅ 油价数据清洗完毕 (缺失值已填补)
⏳ 正在合并 Train 和 Test...
⏳ 正在关联 Store 和 Oil 信息...
⏳ 正在进行最终内存优化...
📉 原始内存占用: 277.35 MB
✅ 优化后内存占用: 179.12 MB
🚀 内存减少了: 35.4%

🎉 阶段1完成！最终数据形状: (3029400, 12)


/var/folders/s0/10htrxbs5_q9tv6b9b30ry4r0000gn/T/ipykernel_11580/2287320678.py:36: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/opt/anaconda3/envs/ai6102/lib/python3.13/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,id,date,store_nbr,family,sales,onpromotion,type_x,city,state,type_y,cluster,oil_price
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,train,Quito,Pichincha,D,13,93.125
1,1,2013-01-01,1,BABY CARE,0.0,0,train,Quito,Pichincha,D,13,93.125
2,2,2013-01-01,1,BEAUTY,0.0,0,train,Quito,Pichincha,D,13,93.125
3,3,2013-01-01,1,BEVERAGES,0.0,0,train,Quito,Pichincha,D,13,93.125
4,4,2013-01-01,1,BOOKS,0.0,0,train,Quito,Pichincha,D,13,93.125


In [17]:
import os

# 1. 创建一个专门存放"中间产物"的文件夹
# 好的工程习惯：不要把加工后的数据和原始数据混在一起
OUTPUT_DIR = 'processed_data'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"📂 创建文件夹成功: {OUTPUT_DIR}")

# 2. 保存为 Pickle 格式 (保留了所有数据类型和优化)
save_path = os.path.join(OUTPUT_DIR, 'df_step1_cleaned.pkl')
print("⏳ 正在保存文件 (这可能需要几秒钟)...")

df.to_pickle(save_path)

print(f"✅ 保存成功！文件位于: {save_path}")
print("👉 你可以将这个 .pkl 文件发送给队友，或者在下一阶段直接读取它。")

⏳ 正在保存文件 (这可能需要几秒钟)...
✅ 保存成功！文件位于: processed_data/df_step1_cleaned.pkl
👉 你可以将这个 .pkl 文件发送给队友，或者在下一阶段直接读取它。
